# Exercise 3: Capacity Expansion Planning

**Presenter** <br>
Priyesh Gosai <br>
Energy Systems Modeller <br>
priyesh@innovateimpact.com <br>

**Course Details**<br>
Modelling Integrated Power Markets <br>
Johannesburg, South Africa <br>
20-24 April 2026 <br>


Name

In [1]:
lesson_number = 3
print(f'lesson{lesson_number}')

lesson3


In [ ]:
#@title Connect to Google Drive {display-mode:"form"}
CONNECT_TO_DRIVE = False #@param {type:"boolean"}

import os

if CONNECT_TO_DRIVE:
    from google.colab import drive
    # Mount Google Drive
    drive.mount('/content/drive')

    # Define the desired working directory path
    working_dir = f'/content/drive/MyDrive/ich-modelling-2026'
    lesson_folder = f'lesson-{lesson_number}'
    lesson_dir = os.path.join(working_dir, lesson_folder)

    # Create the working directory if it doesn't exist
    if not os.path.exists(working_dir):
        os.makedirs(working_dir)
        print(f"Directory '{working_dir}' created.")
    else:
        print(f"Directory '{working_dir}' already exists.")

    # Create the lesson directory if it doesn't exist
    if not os.path.exists(lesson_dir):
        os.makedirs(lesson_dir)
        print(f"Directory '{lesson_dir}' created.")
    else:
        print(f"Directory '{lesson_dir}' already exists.")

    # Change the current working directory to the lesson directory
    os.chdir(lesson_dir)

    print(f"Current working directory: {os.getcwd()}")
else:
    print("Not connecting to Google Drive.")

In [ ]:
#@title Install Packages {display-mode:"form"}
INSTALL_PACKAGES = False #@param {type:"boolean"}

import os

# Check if packages have already been installed in this session to prevent re-installation
if INSTALL_PACKAGES and not os.environ.get('PYPSA_PACKAGES_INSTALLED'):
  !pip install git+https://github.com/pypsa/pypsa
  !pip install pypsa[excel] 
  !pip install folium mapclassify cartopy gurobipy
  os.environ['PYPSA_PACKAGES_INSTALLED'] = 'true'
elif not INSTALL_PACKAGES:
  print("Skipping package installation.")
else:
  print("PyPSA packages are already installed for this session.")

In [ ]:
#@title Download the file for this notebook {display-mode:"form"}
DOWNLOAD_FILE = False #@param {type:"boolean"}

import urllib.request
import os

if DOWNLOAD_FILE:
    url = "https://raw.githubusercontent.com/PriyeshGosai/ich_course_2026/46e109dd68c635d83046eec03d2d2a4afc669366/n-01-single-node-v2.xlsx"
    filename = url.split('/')[-1]  # Extract filename from URL
    
    urllib.request.urlretrieve(url, filename)
    print(f"File downloaded successfully: {os.path.abspath(filename)}")

else:
    print("Skipping file download.")

## Functions

In [ ]:
def setup_gurobi_license_file():
    from google.colab import userdata
    import os
    
    wls_access_id = userdata.get('WLSACCESSID')
    wls_secret = userdata.get('WLSSECRET')
    license_id = userdata.get('LICENSEID')
    
    license_content = f"""WLSACCESSID={wls_access_id}
                        WLSSECRET={wls_secret}
                        LICENSEID={license_id}
                        """
    
    os.makedirs('/opt/gurobi', exist_ok=True)
    with open('/opt/gurobi/gurobi.lic', 'w') as f:
        f.write(license_content)
    
    print("✓ License file created")
    print(f"  License ID: {license_id}")


In [ ]:
def calculate_generator_co2(n):
    """
    Calculate total and per-generator CO2 emissions from actual power output.
    
    Parameters
    ----------
    n : pypsa.Network
        The network object
    
    Returns
    -------
    total_co2 : float
        Total CO2 emissions in tonnes
    df_co2 : pd.DataFrame
        DataFrame with CO2 per generator including efficiency, emissions factor, and p_nom
    """
    import pandas as pd
    
    gen_power = n.generators.dynamic.p  # Power output over time
    gen_co2_by_generator = {}
    
    for gen_name in gen_power.columns:
        carrier = n.generators.static.loc[gen_name, 'carrier']
        co2_rate = n.carriers.static.loc[carrier, 'co2_emissions']
        efficiency = n.generators.static.loc[gen_name, 'efficiency']
        p_nom = n.generators.static.loc[gen_name, 'p_nom']
        
        # Sum actual power output and multiply by CO2 rate, adjusted for efficiency
        gen_co2 = gen_power[gen_name].sum() * co2_rate / efficiency
        
        gen_co2_by_generator[gen_name] = {
            'co2_emissions': gen_co2,
            'efficiency': efficiency,
            'emissions_factor': co2_rate,
            'p_nom': p_nom
        }
    
    # Convert to DataFrame
    df_co2 = pd.DataFrame(gen_co2_by_generator).T
    df_co2 = df_co2.sort_values('co2_emissions', ascending=False)
    
    total_co2 = df_co2['co2_emissions'].sum()
    
    return total_co2, df_co2



In [ ]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pandas as pd
from datetime import datetime

# ============================================================================
# MATPLOTLIB VERSION
# ============================================================================

def plot_dispatch_matplotlib(n, start_date=None, end_date=None, aggregate_data=True):
    """
    Plot dispatch with Matplotlib.
    
    Parameters:
    -----------
    n : pypsa.Network
        The PyPSA network object containing the data to plot.
    start_date : str, optional
        Start date in format 'YYYY-MM-DD' (always 00:00)
    end_date : str, optional
        End date in format 'YYYY-MM-DD' (always 23:00)
    aggregate_data : bool, default True
        If True, aggregate all generators/storage units.
        If False, show individual plant contributions.
    """
    
    # 1. Get generator names (excluding elastic demand)
    non_elastic_generators = n.generators.static[n.generators.static.sign >= 0].index
    
    # 2. Split dispatch and storage
    gen_dispatch = n.generators.dynamic.p[non_elastic_generators]  # Exclude elastic demand
    storage_discharge = n.storage_units.dynamic.p[n.storage_units.dynamic.p >= 0]
    storage_charge = n.storage_units.dynamic.p[n.storage_units.dynamic.p < 0].abs()

    # 3. Prepare data - convert charging to negative
    charging = -storage_charge.fillna(0)  # Negative for below x-axis
    generators = gen_dispatch.fillna(0)
    discharging = storage_discharge.fillna(0)

    # 4. Get load data
    fixed_load = n.loads.dynamic.p.sum(axis=1)
    elastic_generators = n.generators.static[n.generators.static.sign < 0].index
    elastic_load = n.generators.dynamic.p[elastic_generators].sum(axis=1)

    # Filter data by date range
    if start_date and end_date:
        start_dt = pd.Timestamp(start_date + ' 00:00')
        end_dt = pd.Timestamp(end_date + ' 23:00')
        
        mask = (charging.index >= start_dt) & (charging.index <= end_dt)
        charging_filtered = charging[mask]
        generators_filtered = generators[mask]
        discharging_filtered = discharging[mask]
        fixed_load_filtered = fixed_load[mask]
        elastic_load_filtered = elastic_load[mask]
    else:
        charging_filtered = charging
        generators_filtered = generators
        discharging_filtered = discharging
        fixed_load_filtered = fixed_load
        elastic_load_filtered = elastic_load
    
    # Create figure
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Stacked area plot
    if aggregate_data:
        # Aggregated view
        ax.stackplot(
            charging_filtered.index,
            charging_filtered.sum(axis=1),  # Negative
            generators_filtered.sum(axis=1),
            discharging_filtered.sum(axis=1),
            labels=['Charging Storage Units', 'Generator Dispatch', 'Discharging Storage Units'],
            alpha=0.7,
            colors=['#1f77b4', '#ff7f0e', '#2ca02c']
        )
    else:
        # Individual plant view
        # Charging storage units (negative)
        charging_sum = charging_filtered.sum(axis=1)
        ax.fill_between(charging_filtered.index, 0, charging_sum, alpha=0.7, color='#1f77b4', label='Charging Storage Units')
        
        # Generator dispatch (individual)
        ax.stackplot(
            generators_filtered.index,
            *[generators_filtered[col] for col in generators_filtered.columns],
            alpha=0.7,
            labels=generators_filtered.columns
        )
        
        # Discharging storage units (individual, stacked above generators)
        gen_cumsum = generators_filtered.sum(axis=1)
        for col in discharging_filtered.columns:
            ax.fill_between(discharging_filtered.index, gen_cumsum, gen_cumsum + discharging_filtered[col], 
                           alpha=0.7, label=col)
            gen_cumsum = gen_cumsum + discharging_filtered[col]
    
    # Overlay loads
    load_total = fixed_load_filtered + elastic_load_filtered
    ax.plot(fixed_load_filtered.index, fixed_load_filtered, color='black', linewidth=2.5, label='Total Fixed Load', zorder=5)
    ax.plot(load_total.index, load_total, color='red', linewidth=2.5, linestyle='--', label='Total Load (Fixed + Elastic)', zorder=5)
    
    # Formatting
    ax.axhline(y=0, color='black', linewidth=0.8, zorder=4)
    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Power (MW)', fontsize=12)
    ax.set_title('Dispatch with Storage Charging/Discharging and Load', fontsize=14)
    ax.set_xlim(charging_filtered.index.min(), charging_filtered.index.max())
    ax.legend(loc='upper left', fontsize=9, bbox_to_anchor=(1.01, 1))
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


# ============================================================================
# PLOTLY VERSION
# ============================================================================

def plot_dispatch_plotly(n, aggregate_data=True):
    """
    Plot dispatch with Plotly (interactive).
    
    Parameters:
    -----------
    n : pypsa.Network
        The PyPSA network object containing the data to plot.
    aggregate_data : bool, default True
        If True, aggregate all generators/storage units.
        If False, show individual plant contributions.
    """
    
    # 1. Get generator names (excluding elastic demand)
    non_elastic_generators = n.generators.static[n.generators.static.sign >= 0].index
    
    # 2. Split dispatch and storage
    gen_dispatch = n.generators.dynamic.p[non_elastic_generators]  # Exclude elastic demand
    storage_discharge = n.storage_units.dynamic.p[n.storage_units.dynamic.p >= 0]
    storage_charge = n.storage_units.dynamic.p[n.storage_units.dynamic.p < 0].abs()

    # 3. Prepare data - convert charging to negative
    charging = -storage_charge.fillna(0)  # Negative for below x-axis
    generators = gen_dispatch.fillna(0)
    discharging = storage_discharge.fillna(0)

    # 4. Get load data
    fixed_load = n.loads.dynamic.p.sum(axis=1)
    elastic_generators = n.generators.static[n.generators.static.sign < 0].index
    elastic_load = n.generators.dynamic.p[elastic_generators].sum(axis=1)

    fig = go.Figure()
    
    if aggregate_data:
        # Aggregated view
        fig.add_trace(go.Scatter(
            x=charging.index,
            y=charging.sum(axis=1),
            mode='lines',
            name='Charging Storage Units',
            stackgroup='one',
            fillcolor='rgba(31, 119, 180, 0.7)',
            line=dict(color='rgba(31, 119, 180, 0.7)')
        ))
        
        fig.add_trace(go.Scatter(
            x=generators.index,
            y=generators.sum(axis=1),
            mode='lines',
            name='Generator Dispatch',
            stackgroup='one',
            fillcolor='rgba(255, 127, 14, 0.7)',
            line=dict(color='rgba(255, 127, 14, 0.7)')
        ))
        
        fig.add_trace(go.Scatter(
            x=discharging.index,
            y=discharging.sum(axis=1),
            mode='lines',
            name='Discharging Storage Units',
            stackgroup='one',
            fillcolor='rgba(44, 160, 44, 0.7)',
            line=dict(color='rgba(44, 160, 44, 0.7)')
        ))
    else:
        # Individual plant view
        # Charging storage units (negative, non-stacking)
        for col in charging.columns:
            fig.add_trace(go.Scatter(
                x=charging.index,
                y=charging[col],
                mode='lines',
                name=f'{col} (Charging)',
                stackgroup='charging',
                fillcolor='rgba(31, 119, 180, 0.7)',
                line=dict(color='rgba(31, 119, 180, 0.7)')
            ))
        
        # Generator dispatch (individual, stacked positive)
        for col in generators.columns:
            fig.add_trace(go.Scatter(
                x=generators.index,
                y=generators[col],
                mode='lines',
                name=col,
                stackgroup='generation',
                fillcolor=None,
                line=dict(width=1)
            ))
        
        # Discharging storage units (individual, stacked positive)
        for col in discharging.columns:
            fig.add_trace(go.Scatter(
                x=discharging.index,
                y=discharging[col],
                mode='lines',
                name=f'{col} (Discharging)',
                stackgroup='generation',
                fillcolor=None,
                line=dict(width=1)
            ))
    
    # Overlay loads
    fig.add_trace(go.Scatter(
        x=fixed_load.index,
        y=fixed_load,
        mode='lines',
        name='Total Fixed Load',
        line=dict(color='black', width=2.5),
        yaxis='y'
    ))
    
    load_total = fixed_load + elastic_load
    fig.add_trace(go.Scatter(
        x=load_total.index,
        y=load_total,
        mode='lines',
        name='Total Load (Fixed + Elastic)',
        line=dict(color='red', width=2.5, dash='dash'),
        yaxis='y'
    ))
    
    # Layout
    fig.update_layout(
        title='Dispatch with Storage Charging/Discharging and Load',
        xaxis_title='Time',
        yaxis_title='Power (MW)',
        hovermode='x unified',
        height=600,
        template='plotly_white'
    )
    
    fig.show()

## Model

#### Case Description

Perform a capacity expansion plan for solar, wind and new gas generators with build year >= 2026 and determine the optimal capacity given the available profiles. Due to the computational complexity, we will resample the network from an hourly resolution to four-hour resolution.

##### PyPSA Features Used

| Feature | Method |
|---------|--------|
| Organising code | Inspect notebook |
| Adding metadata | From an Excel sheet |
| Filtering pandas dataframes | Programmatically |
| Changing values in DataFrames programmatically | Programmatically |
| Using the `p_nom_extendable` variable | Programmatically |
| Constraining extended `p_nom` limits within bounds | Programmatically |
| Clustering network | Programmatically |
| Calculating Emissions factor | From Excel sheet |
| Setting Global Constraints | Programmatically |
| Using discount rates, overnight costs, FOM costs <br> and lifetime variables to calculate capital cost | Inspect Excel file |
| Using the statistics library to analyse results | Programmatically |
| Creating custom plots | Programmatically |

##### Non-Standard PyPSA Features

These features are not standard in PyPSA but can be implemented through component manipulation:

| Feature | Method |
|---------|--------|
| Setting up loadshedding generator | Inspect Excel file |
| Accounting for price elastic demand | Inspect Excel file |

### Lesson Task

Evaluate the optimal dispatch results with and without a $CO_2$ constraint by programatically setting the limit to 

* 1,500,000 tonne $CO_2$
* 2,500,000 tonne $CO_2$
* 5,000,000 tonne $CO_2$


#### Optional Tasks

Co-optimize the storage with the other generators — This is a computationally expensive task and could take quite long to solve. Consider using Gurobi.

_Note: The model takes approximately 3 minutes to solve. It would be helpful to work in groups to look at different scenarios._

### User inputs

In [ ]:
file_name = 'n-01-single-node-v2.xlsx'
solver_name = 'gurobi'  # 'gurobi' or 'highs'


### Import packages, read input data and optimize

In [ ]:
# Import packages
import gurobipy as gp
import pypsa
import pandas as pd
import linopy
import plotly.graph_objects as go

import linopy.solvers
linopy.solvers.gurobipy = gp

# Supress unnecessary warnings
pd.set_option('future.no_silent_downcasting', True)

# The default plotting tool for pandas is matplotlib which only provies static plots. 
# Changing to plotly allows for interactive plotting. 
pd.set_option('plotting.backend', 'plotly')

# When pypsa version 1 was launched, a new set of commands were created. 
# This option ensures that we use the new commands. 
pypsa.options.api.new_components_api = True

In [ ]:
# Model import and optimization
print(f'Load Network from {file_name}')
n = pypsa.Network(file_name)

# Import metadata (optional)
n.meta = pd.read_excel(file_name, sheet_name='meta', index_col=0, header=None)[1].to_dict()

In [ ]:
# n.cluster.temporal.downsample(4)


n.generators.static.loc[n.generators.static.build_year >= 2026, 'p_nom_extendable'] = True
n.storage_units.static.loc[n.storage_units.static.build_year >= 2026, 'p_nom_extendable'] = True 


n.generators.static.loc[(n.generators.static.build_year >= 2026) & (n.generators.static.carrier == 'solar'), 'p_nom_max'] = 1500
# n.generators.static.loc[(n.generators.static.build_year >= 2026) & (n.generators.static.carrier == 'wind'), 'p_nom_max'] = 800
n.generators.static.loc[(n.generators.static.build_year >= 2026) & (n.generators.static.carrier == 'gas'), 'p_nom_max'] = 1200
n.storage_units.static.loc[n.storage_units.static.build_year >= 2026 , 'p_nom_max'] = 1000

n.generators.static.loc['Loadshedding WC','active'] = False
n.generators.static.loc['Loadshedding WC','marginal_cost'] = 50000

n.generators.static.loc['Cape Town Elastic Demand 1','active'] = False
n.generators.static.loc['Cape Town Elastic Demand 2','active'] = False
n.generators.static.loc['Cape Town Elastic Demand 3','active'] = False
n.generators.static.loc['Cape Town Elastic Demand 4','active'] = False
n.generators.static.loc['Cape Town Elastic Demand 5','active'] = False



# co2_cap = 3.6e6
# n.add("GlobalConstraint", "CO2Limit", type="primary_energy", carrier_attribute="co2_emissions", sense="<=", constant=co2_cap)

In [ ]:
# Create model
n.optimize.create_model()

# Add wind capacity constraint
wind_mask = (n.generators.static['carrier'] == 'wind') & \
            (n.generators.static['build_year'] >= 2026)
wind_gens = n.generators.static[wind_mask].index

# Use .loc to select by index labels
wind_capacity_constraint = n.model.variables["Generator-p_nom"].loc[wind_gens].sum() <= 4000

# Add to network
n.model.add_constraints(wind_capacity_constraint, name='wind_capacity_limit')

# Solve
n.optimize.solve_model(solver_name=solver_name)

print(f"Affected wind generators: {len(wind_gens)}")
print(f"Total wind capacity: {n.generators.static.loc[wind_gens, 'p_nom_opt'].sum():.2f} MW")
print(wind_gens.tolist())

In [ ]:
# Run optimization

print('Run optimization')
n.optimize(include_objective_constant = True, solver_name=solver_name)

In [ ]:
# Optimal capacity per generator
print("Generator Capacities:")
print(n.generators.static[['carrier', 'p_nom_opt']])
print(f"\nTotal generator capacity: {n.generators.static['p_nom_opt'].sum():.2f} MW")

# Optimal capacity per storage unit
print("\n\nStorage Unit Capacities:")
print(n.storage_units.static[['carrier', 'p_nom_opt']])
print(f"\nTotal storage capacity: {n.storage_units.static['p_nom_opt'].sum():.2f} MW")




In [ ]:
capex = n.statistics.capex() # returns a series
capex.to_frame("capex")

In [ ]:
system_cost = n.statistics.system_cost()
print(f"Total system cost: {system_cost.sum() / 1e9:.1f} bn ZAR")

In [ ]:
n.generators.dynamic.p.plot()

In [ ]:
costs = pd.concat([
    n.statistics.capex(groupby_method="sum").rename("sum"),
    n.statistics.capex(groupby_method="mean").rename("mean"),
], axis=1)
    
costs.sort_values(by="sum", ascending=False).div(1e9).style.format("{:,.0f} bn ZAR")

In [ ]:
caps = n.statistics.optimal_capacity()
caps.to_frame().head(10)

In [ ]:
cf = n.statistics.capacity_factor()
cf.sort_values(ascending=False).dropna()

In [ ]:
n.statistics.supply().sort_values(ascending=False).head(10)

In [ ]:
n.statistics.withdrawal().sort_values().head(10)

In [ ]:
eb = n.statistics.energy_balance()
eb.head(15)

In [ ]:
curt = n.statistics.curtailment()
curt[curt > 0].sort_values(ascending=False)

In [ ]:
n.statistics.revenue().sort_values(ascending=False).head(10)

In [ ]:
df_opt = n.generators.static.p_nom_opt

display(df_opt)

In [ ]:
n.storage_units.static.p_nom_opt

### Dispatch results

In [ ]:
n.generators.dynamic.p.plot()

In [ ]:
n.storage_units.dynamic.p.plot()

### Custom plots

Inspect functions at the top of the notebook. 

In [ ]:
# Interactive plot with Plotly

plot_dispatch_plotly(n,aggregate_data=False)

In [ ]:
# Static plot with Matplotlib (for all data or a specific date range)
plot_dispatch_matplotlib(n)  # All data


In [ ]:
plot_dispatch_matplotlib(n,start_date='2026-01-01', end_date='2026-01-05',aggregate_data=False)  # Specific date range, individual plants

In [ ]:
total_co2, df_co2 = calculate_generator_co2(n)
print(f"Total CO2 from actual generation: {total_co2:.2f} tonne CO2")
print("\nCO2 per generator:")
print(df_co2)